# SMARD: Machine Learning

Reads `data/interim/clean_hourly.csv`, produced by the Clean section of `SMARD_analysis.ipynb`; the Clean, EDA, Visualizations, and Insights sections live in that notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

from src import config, eda, models, evaluate, visualize as viz

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [ ]:
df = pd.read_csv(config.CLEAN_FILE, index_col=0, parse_dates=True)
assert len(df) == 17544, f"expected 17544 rows, got {len(df)}"

In [ ]:
# Duplicated from the EDA section of SMARD_analysis.ipynb -- keep the two lists in sync.
CANDIDATES = [
    "load", "Residual load", "net_residual_load", "renewable_gen",
    "vre_gen", "wind_total", "Wind onshore", "Wind offshore",
    "Photovoltaics", "renewable_share", "vre_share", "conventional_gen",
    "Fossil gas", "Hard coal", "Lignite", "Nuclear", "Biomass",
    "Hydropower", "Pumped storage consumption", "total_gen",
    "price_lag24", "price_lag168", "load_lag24",
    "hour", "dayofweek", "month", "is_weekend",
]

## Machine learning

The dataset holds **actual** generation and load, not day-ahead forecasts. Models trained on actuals are **explanatory / attribution**, not forecasting. Only the model restricted to lags and calendar features (`sets["forecast"]`) is forecast-legal. The primary model is the negative-price classifier; the GBM/Ridge regressors on actuals are secondary, attribution-only.

All hyperparameters below are pinned in `src/models.py` (`FORECAST_GBM_MAX_ITER`, `ATTRIBUTION_GBM_MAX_ITER`, `RIDGE_ALPHA`, `CLASSIFIER_MAX_ITER`), selected on a chronological validation split *within* 2024 (train Jan–Sep, validate Oct–Dec), with 2025 used exactly once, for final evaluation. Nothing here re-tunes them.

In [ ]:
from src import models, evaluate
import numpy as np

df = pd.read_csv(config.CLEAN_FILE, index_col=0, parse_dates=True)
assert len(df) == 17544, f"expected 17544 rows, got {len(df)}"

### Feature sets

In [ ]:
surviving, _ = eda.screen_candidates(df, CANDIDATES)
sets = models.build_feature_sets(df, surviving)
kept, pruned = models.prune_collinear(df, sets["explanatory"], priority=None)
print(f"forecast set: {len(sets['forecast'])} | explanatory: {len(kept)} after pruning {len(pruned)}")
pruned

### Chronological split

In [ ]:
train, test = evaluate.chronological_split(df)
print(f"train {len(train)} ({train.index.min().date()} to {train.index.max().date()})")
print(f"test  {len(test)} ({test.index.min().date()} to {test.index.max().date()})")

### Primary model: negative-price classifier

Base rate is ~5.9%, so PR-AUC (`average_precision`) leads; accuracy alone is uninformative at this imbalance.

In [ ]:
clf = models.train_negative_price_classifier(train, kept)
clf_prob = pd.Series(clf.predict_proba(test[kept])[:, 1], index=test.index)
evaluate.classification_metrics(test["is_negative_price"], clf_prob)

In [ ]:
evaluate.threshold_sweep(test["is_negative_price"], clf_prob)

*Reading:*

### Secondary model: price attribution (NOT a forecast)

In [ ]:
attribution_gbm = models.train_attribution_gbm(train, kept)
pred_actuals = pd.Series(attribution_gbm.predict(test[kept]), index=test.index)
evaluate.regression_metrics(test["price"], pred_actuals)

### Honest forecast: lags and calendar only

In [ ]:
forecast_gbm = models.train_forecast_gbm(train, sets["forecast"])
pred_forecast = pd.Series(forecast_gbm.predict(test[sets["forecast"]]), index=test.index)

ridge = models.train_ridge_baseline(train, kept)
test_ridge = test.dropna(subset=["price"] + kept)
pred_ridge = pd.Series(ridge.predict(test_ridge[kept]), index=test_ridge.index)

pred_naive = evaluate.naive_baseline(test)

evaluate.regression_metrics(test["price"], pred_forecast)

### Model comparison

In [ ]:
results = {
    "naive": pred_naive,
    "forecast_gbm": pred_forecast,
    "attribution_gbm": pred_actuals,
    "ridge": pred_ridge,
}
evaluate.compare_models(results, test["price"])

### Error by price regime

In [ ]:
evaluate.metrics_by_regime(test["price"], pred_actuals)

*Reading:*

### Feature attribution (permutation importance)

Collinear features were pruned first (`prune_collinear`, above), so importance below is not split between correlated twins. Importance is measured on the 2025 test set, so it reflects out-of-sample attribution across the 2024→2025 regime shift, not training-set fit.

In [ ]:
imp = models.permutation_feature_importance(attribution_gbm, test[kept], test["price"])
imp

In [ ]:
fig = viz.plot_permutation_importance(imp); fig

In [ ]:
top3 = imp["feature"].head(3).tolist()
pdp = models.partial_dependence_data(attribution_gbm, test[kept], top3)
top3

In [ ]:
fig = viz.plot_partial_dependence(pdp, top3[0]); fig

In [ ]:
fig = viz.plot_partial_dependence(pdp, top3[1]); fig

In [ ]:
fig = viz.plot_partial_dependence(pdp, top3[2]); fig

*Reading:*

### Benchmark reproduction

In [ ]:
naive_metrics = evaluate.regression_metrics(test["price"], pred_naive)
assert abs(naive_metrics["mae"] - 25.97) < 0.1, f"naive MAE is {naive_metrics['mae']:.3f}, expected ~25.97"
print(f"PASS: naive MAE == {naive_metrics['mae']:.3f}")

# HistGradientBoosting results are sklearn-version-dependent, so the GBM tolerances below are
# loose (1.5 MAE / 0.05 R2). naive and Ridge use tight tolerances since they are stable across
# sklearn versions.

forecast_metrics = evaluate.regression_metrics(test["price"], pred_forecast)
assert abs(forecast_metrics["mae"] - 23.70) < 1.5, f"honest GBM MAE is {forecast_metrics['mae']:.3f}, expected ~23.70"
assert abs(forecast_metrics["r2"] - 0.540) < 0.05, f"honest GBM R2 is {forecast_metrics['r2']:.3f}, expected ~0.540"
print(f"PASS: honest GBM MAE == {forecast_metrics['mae']:.3f}, R2 == {forecast_metrics['r2']:.3f}")

attribution_metrics = evaluate.regression_metrics(test["price"], pred_actuals)
assert abs(attribution_metrics["mae"] - 15.39) < 1.5, f"attribution GBM MAE is {attribution_metrics['mae']:.3f}, expected ~15.39"
assert abs(attribution_metrics["r2"] - 0.789) < 0.05, f"attribution GBM R2 is {attribution_metrics['r2']:.3f}, expected ~0.789"
print(f"PASS: attribution GBM MAE == {attribution_metrics['mae']:.3f}, R2 == {attribution_metrics['r2']:.3f}")

ridge_metrics = evaluate.regression_metrics(test_ridge["price"], pred_ridge)
assert abs(ridge_metrics["mae"] - 16.92) < 0.5, f"Ridge MAE is {ridge_metrics['mae']:.3f}, expected ~16.92"
print(f"PASS: Ridge MAE == {ridge_metrics['mae']:.3f}, R2 == {ridge_metrics['r2']:.3f}")

clf_metrics = evaluate.classification_metrics(test["is_negative_price"], clf_prob)
assert abs(clf_metrics["average_precision"] - 0.784) < 0.05, f"classifier average_precision is {clf_metrics['average_precision']:.3f}, expected ~0.784"
print(f"PASS: classifier average_precision == {clf_metrics['average_precision']:.3f}")

assert forecast_metrics["mae"] < naive_metrics["mae"], (
    f"honest GBM MAE ({forecast_metrics['mae']:.3f}) should be lower than naive "
    f"({naive_metrics['mae']:.3f}) -- a forecast-legal model should beat persistence"
)
print(f"PASS: honest, forecast-legal GBM (MAE {forecast_metrics['mae']:.3f}) beats naive persistence (MAE {naive_metrics['mae']:.3f})")

## Machine learning complete

_Power BI export comes next._